<a href="https://colab.research.google.com/github/yuhui-0611/ESAA/blob/main/ESAA_OB_mini_project2_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 소설 작가 분류 AI 경진대회

- 본 알고리즘은 작가별 문체 차이를 반영하기 위해 단어 기반 TF-IDF, 문자 기반 TF-IDF, 문장 구조 기반 수치 특징을 함께 사용
- 단어 TF-IDF는 작가가 자주 사용하는 어휘와 표현을 포착하고, 문자 TF-IDF는 접미사, 축약형, 구두점 사용 등 세밀한 문체 패턴을 반영
- 또한 문장 길이, 평균 단어 길이, 쉼표·마침표·따옴표 사용 빈도 등의 수치 특징을 추가하여 문체적 습관을 보완적으로 학습
- 최종적으로 이 특징들을 결합한 뒤 모델을 학습하여 각 문장이 어느 작가의 문체에 가까운지 확률 형태로 예측

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer

In [3]:
X_train = pd.read_csv('/content/drive/MyDrive/contest/text_analyze/train.csv', encoding = 'utf-8')
X_test = pd.read_csv('/content/drive/MyDrive/contest/text_analyze/test_x.csv', encoding = 'utf-8')

In [5]:
Y_train = LabelEncoder().fit_transform(X_train['author'])
Y_train

array([3, 2, 1, ..., 1, 3, 0])

In [4]:
X_train['author'].value_counts()

,count
author,
3,15063
0,13235
2,11554
4,7805
1,7222


- authors = ['0','1','2','3','4']

## preprocessing

In [6]:
import re

def clean(X_train,X_test):
    X_train['words'] = [re.sub("[^a-zA-Z]"," ", data).lower().split() for data in X_train['text']]
    X_test['words'] = [re.sub("[^a-zA-Z]"," ", data).lower().split() for data in X_test['text']]
    return X_train,X_test

In [7]:
X_train, X_test = clean(X_train, X_test)

- .split() : 공백 기준 단어 분리
- text의 특수문자 제거 > 소문자화 > 단어 분리

## feature engineering

### Punctuation

In [8]:
punctuations = [
    {"id":1,"p":"[;:]"},
    {"id":2,"p":"[,.]"},
    {"id":3,"p":"[?]"},
    {"id":4,"p":"[\']"},
    {"id":5,"p":"[\"]"},
    {"id":6,"p":"[;:,.?\'\"]"}
]

In [9]:
for p in punctuations:
    punctuation = p["p"]
    _train =  [ sentence.split() for sentence in X_train['text'] ]
    X_train['punc_'+str(p["id"])] = [len([
        word for word in sentence
        if bool(re.search(punctuation, word))])*100.0/(len(sentence)+1)  # 비율화
        for sentence in _train]

    _test =  [ sentence.split() for sentence in X_test['text'] ]
    X_test['punc_'+str(p["id"])] = [len([
        word for word in sentence
        if bool(re.search(punctuation, word))])*100.0/(len(sentence)+1)
        for sentence in _test]

- len([...]) : 구두점 포함 단어 개수 계산 = 해당 구두점 사용 횟수
> 글쓰기 style 수치화

In [10]:
X_train.head()

,index,text,author,words,punc_1,punc_2,punc_3,punc_4,punc_5,punc_6
0,0,"He was almost choking. There was so much, so m...",3,"[he, was, almost, choking, there, was, so, muc...",2.127660,14.893617,0.0,0.000000,0.0,17.021277
1,1,"“Your sister asked for it, I suppose?”",2,"[your, sister, asked, for, it, i, suppose]",0.000000,12.500000,12.5,0.000000,0.0,25.000000
2,2,"She was engaged one day as she walked, in per...",1,"[she, was, engaged, one, day, as, she, walked,...",1.724138,13.793103,0.0,0.000000,0.0,15.517241
3,3,"The captain was in the porch, keeping himself ...",4,"[the, captain, was, in, the, porch, keeping, h...",3.389831,25.423729,0.0,1.694915,0.0,30.508475
4,4,"“Have mercy, gentlemen!” odin flung up his han...",3,"[have, mercy, gentlemen, odin, flung, up, his,...",2.500000,17.500000,0.0,0.000000,0.0,20.000000


### Stop Words

- 일반 NLP에서는 stopword를 제거
- 하지만 문제 분석이기에 stopword 제거 X
  - 작가마다 문체 스타일이 다르기 때문

In [11]:
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [12]:
stop_words = set(stopwords.words('english'))

X_train['stop_word'] = [
    len([word for word in sentence if word in stop_words])
    * 100.0 / (len(sentence)+1)

    for sentence in X_train['words']
]

X_test['stop_word'] = [
    len([word for word in sentence if word in stop_words])
    * 100.0 / (len(sentence)+1)

    for sentence in X_test['words']
]

### statistical style features

- 작가의 통계적 문체 특징 코드
- 긴 문장을 쓰는지, 다양한 단어를 쓰는지, 대문자를 많이 쓰는지, 대화체가 많은지 등

In [13]:
eng_stopwords = set(stopwords.words("english"))

In [14]:
import string

In [15]:
## Number of words in the text ##
X_train["num_words"] = X_train["text"].apply(lambda x: len(str(x).split()))
X_test["num_words"] = X_test["text"].apply(lambda x: len(str(x).split()))

## Number of unique words in the text ##
X_train["num_unique_words"] = X_train["text"].apply(lambda x: len(set(str(x).split())))
X_test["num_unique_words"] = X_test["text"].apply(lambda x: len(set(str(x).split())))

## Number of characters in the text ##
X_train["num_chars"] = X_train["text"].apply(lambda x: len(str(x)))  # 공백/구두점 포함 문자 개수
X_test["num_chars"] = X_test["text"].apply(lambda x: len(str(x)))

## Number of stopwords in the text ##
X_train["num_stopwords"] = X_train["text"].apply(lambda x: len([
    w for w in str(x).lower().split() if w in eng_stopwords
    ]))
X_test["num_stopwords"] = X_test["text"].apply(lambda x: len([
    w for w in str(x).lower().split() if w in eng_stopwords
    ]))

## Number of punctuations in the text ##
X_train["num_punctuations"] = X_train['text'].apply(lambda x: len([
    c for c in str(x) if c in string.punctuation
    ]))
X_test["num_punctuations"] = X_test['text'].apply(lambda x: len([
    c for c in str(x) if c in string.punctuation
    ]))

## Number of ALL CAPS words in the text ##
X_train["num_words_upper"] = X_train["text"].apply(lambda x: len([
    w for w in str(x).split() if w.isupper()  # 강조 스타일 반영 가능
    ]))
X_test["num_words_upper"] = X_test["text"].apply(lambda x: len([
    w for w in str(x).split() if w.isupper()
    ]))

## Number of Title Case words in the text(첫글자만 대문자) ##
X_train["num_words_title"] = X_train["text"].apply(lambda x: len([
    w for w in str(x).split() if w.istitle()
    ]))
X_test["num_words_title"] = X_test["text"].apply(lambda x: len([
    w for w in str(x).split() if w.istitle()
    ]))

## Average length of the words in the text ##
X_train["mean_word_len"] = X_train["text"].apply(lambda x: np.mean([
    len(w) for w in str(x).split()
    ]))
X_test["mean_word_len"] = X_test["text"].apply(lambda x: np.mean([
    len(w) for w in str(x).split()
    ]))

| feature          | 의미        |
| ---------------- | --------- |
| num_words        | 문장 길이     |
| num_unique_words | 어휘 다양성    |
| num_chars        | 전체 문자 길이  |
| num_stopwords    | 기능어 사용    |
| num_punctuations | 구두점 스타일   |
| num_words_upper  | 강조 스타일    |
| num_words_title  | 제목형 단어 사용 |
| mean_word_len    | 평균 단어 복잡도 |


### TF-IDF

In [16]:
from sklearn import preprocessing, decomposition, model_selection, metrics, pipeline
from sklearn import ensemble, metrics, model_selection, naive_bayes

**< word - 단어단위 >**

In [17]:
def tfidfWords(X_train,X_test):
    tfidf_vec = TfidfVectorizer(stop_words='english', ngram_range=(1,2))
    full_tfidf = tfidf_vec.fit_transform(X_train['text'].values.tolist())
    train_tfidf = tfidf_vec.transform(X_train['text'].values.tolist())  # 학습 데이터의 TF-IDF 행렬
    test_tfidf = tfidf_vec.transform(X_test['text'].values.tolist())  # 테스트 데이터의 TF-IDF 행렬
    return train_tfidf,test_tfidf,full_tfidf

- 텍스트를 TF-IDF 숫자 벡터로 변환

In [18]:
def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)  # 각 작가일 확률 예측
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

- Multinomial Naive Bayes 학습
- 텍스트 분류에서 자주 쓰는 모델

In [19]:
def do_tfidf_MNB(X_train, X_test, Y_train):
    train_tfidf, test_tfidf, full_tfidf = tfidfWords(X_train, X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)  # train을 8개으로 나눔
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_tfidf[dev_index], train_tfidf[val_index]  # 학습용
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]  # 검증용
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_tfidf)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.  # 8개 모델의 예측 확률 평균
    return pred_train, pred_full_test

- OOF prediction = out-of-fold prediction
- dev_index, val_index in kf.split(X_train)
  - X_train 데이터를 8개의 fold로 나눈 뒤, 매 반복마다 학습용(dev) index, 검증용(val) index를 반환하는 것

In [20]:
pred_train, pred_test = do_tfidf_MNB(X_train, X_test, Y_train)

Mean cv score :  1.0898864911765824


In [21]:
X_train["tfidf_words_nb_cvec_0"] = pred_train[:,0]
X_train["tfidf_words_nb_cvec_1"] = pred_train[:,1]
X_train["tfidf_words_nb_cvec_2"] = pred_train[:,2]
X_train["tfidf_words_nb_cvec_3"] = pred_train[:,3]
X_train["tfidf_words_nb_cvec_4"] = pred_train[:,4]
X_test["tfidf_words_nb_cvec_0"] = pred_test[:,0]
X_test["tfidf_words_nb_cvec_1"] = pred_test[:,1]
X_test["tfidf_words_nb_cvec_2"] = pred_test[:,2]
X_test["tfidf_words_nb_cvec_3"] = pred_test[:,3]
X_test["tfidf_words_nb_cvec_4"] = pred_test[:,4]

> TF-IDF + Naive Bayes로 1차 작가 단위 분류를 한 뒤, 그 예측 확률 5개를 새로운 feature로 추가하는 코드

**< char - 문자단위 >**

In [22]:
def tfidfWords(X_train,X_test):
    tfidf_vec = TfidfVectorizer(stop_words='english', ngram_range=(3,4), analyzer='char')
    full_tfidf = tfidf_vec.fit_transform(X_train['text'].values.tolist())
    train_tfidf = tfidf_vec.transform(X_train['text'].values.tolist())
    test_tfidf = tfidf_vec.transform(X_test['text'].values.tolist())
    return train_tfidf,test_tfidf

- analyzer='char' : 문자 기준 TF-IDF

In [23]:
def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

In [24]:
def do(X_train,X_test,Y_train):
    train_tfidf,test_tfidf = tfidfWords(X_train,X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_tfidf[dev_index], train_tfidf[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_tfidf)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.
    return pred_train, pred_full_test

In [25]:
pred_train,pred_test = do(X_train,X_test,Y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/feature_extraction/text.py:539: UserWarning: The parameter 'stop_words' will not be used since 'analyzer' != 'word'
  warnings.warn(


Mean cv score :  0.9830868491998358


In [26]:
X_train["tfidf_chars_nb_cvec_0"] = pred_train[:,0]
X_train["tfidf_chars_nb_cvec_1"] = pred_train[:,1]
X_train["tfidf_chars_nb_cvec_2"] = pred_train[:,2]
X_train["tfidf_chars_nb_cvec_3"] = pred_train[:,3]
X_train["tfidf_chars_nb_cvec_4"] = pred_train[:,4]
X_test["tfidf_chars_nb_cvec_0"] = pred_test[:,0]
X_test["tfidf_chars_nb_cvec_1"] = pred_test[:,1]
X_test["tfidf_chars_nb_cvec_2"] = pred_test[:,2]
X_test["tfidf_chars_nb_cvec_3"] = pred_test[:,3]
X_test["tfidf_chars_nb_cvec_4"] = pred_test[:,4]

> TF-IDF + Naive Bayes로 1차 작가 문자 분류를 한 뒤, 그 예측 확률 5개를 새로운 feature로 추가하는 코드

| 방식          | 잘 잡는 것    |
| ----------- | --------- |
| word TF-IDF | 의미/어휘 선택  |
| char TF-IDF | 철자/문체/구두점 |


### Count

| 방식     | 특징       |
| ------ | -------- |
| Count  | 순수 빈도    |
| TF-IDF | 중요 단어 강조(희귀 단어 점수 높임) |


**< word - 단어단위 >**

In [27]:
def countWords(X_train, X_test):
    count_vec = CountVectorizer(stop_words='english', ngram_range=(1,2))
    count_vec.fit(X_train['text'].values.tolist())
    train_count = count_vec.transform(X_train['text'].values.tolist())
    test_count = count_vec.transform(X_test['text'].values.tolist())
    return train_count, test_count

def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

def do_count_MNB(X_train, X_test, Y_train):
    train_count, test_count = countWords(X_train, X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits = 8, shuffle=True, random_state=2020)
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_count[dev_index], train_count[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_count)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.
    return pred_train, pred_full_test

pred_train, pred_test = do_count_MNB(X_train, X_test, Y_train)

X_train["count_words_nb_cvec_0"] = pred_train[:,0]
X_train["count_words_nb_cvec_1"] = pred_train[:,1]
X_train["count_words_nb_cvec_2"] = pred_train[:,2]
X_train["count_words_nb_cvec_3"] = pred_train[:,3]
X_train["count_words_nb_cvec_4"] = pred_train[:,4]
X_test["count_words_nb_cvec_0"] = pred_test[:,0]
X_test["count_words_nb_cvec_1"] = pred_test[:,1]
X_test["count_words_nb_cvec_2"] = pred_test[:,2]
X_test["count_words_nb_cvec_3"] = pred_test[:,3]
X_test["count_words_nb_cvec_4"] = pred_test[:,4]

Mean cv score :  0.9921673388998574


**< word - 단어단위 >**

In [28]:
def countChars(X_train, X_test):
    count_vec = CountVectorizer(ngram_range=(3,4), analyzer='char')
    count_vec.fit(X_train['text'].values.tolist())
    train_count = count_vec.transform(X_train['text'].values.tolist())
    test_count = count_vec.transform(X_test['text'].values.tolist())
    return train_count,test_count

def runMNB(train_X, train_y, test_X, test_y, test_X2):
    model = naive_bayes.MultinomialNB()
    model.fit(train_X, train_y)
    pred_test_y = model.predict_proba(test_X)
    pred_test_y2 = model.predict_proba(test_X2)
    return pred_test_y, pred_test_y2, model

def do_count_chars_MNB(X_train, X_test, Y_train):
    train_count,test_count=countChars(X_train, X_test)
    cv_scores = []
    pred_full_test = 0
    pred_train = np.zeros([X_train.shape[0], 5])
    kf = model_selection.KFold(n_splits=8, shuffle=True, random_state=2020)
    for dev_index, val_index in kf.split(X_train):
        dev_X, val_X = train_count[dev_index], train_count[val_index]
        dev_y, val_y = Y_train[dev_index], Y_train[val_index]
        pred_val_y, pred_test_y, model = runMNB(dev_X, dev_y, val_X, val_y, test_count)
        pred_full_test = pred_full_test + pred_test_y
        pred_train[val_index,:] = pred_val_y
        cv_scores.append(metrics.log_loss(val_y, pred_val_y))
    print("Mean cv score : ", np.mean(cv_scores))
    pred_full_test = pred_full_test / 8.
    return pred_train, pred_full_test

pred_train, pred_test = do_count_chars_MNB(X_train, X_test, Y_train)

X_train["count_chars_nb_cvec_0"] = pred_train[:,0]
X_train["count_chars_nb_cvec_1"] = pred_train[:,1]
X_train["count_chars_nb_cvec_2"] = pred_train[:,2]
X_train["count_chars_nb_cvec_3"] = pred_train[:,3]
X_train["count_chars_nb_cvec_4"] = pred_train[:,4]
X_test["count_chars_nb_cvec_0"] = pred_test[:,0]
X_test["count_chars_nb_cvec_1"] = pred_test[:,1]
X_test["count_chars_nb_cvec_2"] = pred_test[:,2]
X_test["count_chars_nb_cvec_3"] = pred_test[:,3]
X_test["count_chars_nb_cvec_4"] = pred_test[:,4]

Mean cv score :  3.075051208358071


## sentence embedding

- 단어를 의미 벡터(vector)로 바꾸고, 문장 전체를 하나의 벡터로 표현

< GloVe >
- 단어 의미 자체를 숫자 벡터로 표현
- 단어를 숫자 벡터로 바꿔주는 사전
- 단어의 의미를 숫자 100개짜리 좌표로 저장해둔 파일
  - 의미가 비슷한 단어들이 비슷한 벡터를 가짐

In [ ]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

In [29]:
wv = "./glove.6B.100d.txt"

In [30]:
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [31]:
from nltk import word_tokenize
from tqdm import tqdm

In [32]:
# GloVe 사전 불러오는 함수
def loadWordVecs():
    embeddings_index = {}
    f = open(wv)
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
    f.close()
    print('Found %s word vectors.' % len(embeddings_index))
    return embeddings_index

In [33]:
# 정규화 벡터 생성 함수 with GloVe
def sent2vec(embeddings_index, s):
    words = str(s).lower()
    words = word_tokenize(words)
    words = [w for w in words if not w in stopwords.words('english')]
    words = [w for w in words if w.isalpha()]  # 알파벳만 남김
    M = []
    for w in words:
        try:
            M.append(embeddings_index[w])  # 각 단어의 GloVe 벡터 가져오기
        except:
            continue
    M = np.array(M)
    v = M.sum(axis=0)  # 문장 벡터 = 단어 벡터들의 합
    if type(v) != np.ndarray:
        return np.zeros(100)
    return v / np.sqrt((v ** 2).sum())  # 정규화(벡터 길이 1로 맞춤)

In [34]:
def doGlove(x_train, x_test):
    embeddings_index = loadWordVecs()
    xtrain_glove = [sent2vec(embeddings_index,x) for x in tqdm(x_train)]
    xtest_glove = [sent2vec(embeddings_index,x) for x in tqdm(x_test)]
    xtrain_glove = np.array(xtrain_glove)
    xtest_glove = np.array(xtest_glove)
    return xtrain_glove, xtest_glove, embeddings_index

In [35]:
glove_vecs_train, glove_vecs_test, embeddings_index = doGlove(X_train['text'], X_test['text'])
X_train[['sent_vec_'+str(i) for i in range(100)]] = pd.DataFrame(glove_vecs_train.tolist())
X_test[['sent_vec_'+str(i) for i in range(100)]] = pd.DataFrame(glove_vecs_test.tolist())

Found 400000 word vectors.


100%|██████████| 19617/19617 [03:48<00:00, 85.77it/s] 
/tmp/ipykernel_102083/575191149.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[['sent_vec_'+str(i) for i in range(100)]] = pd.DataFrame(glove_vecs_train.tolist())
/tmp/ipykernel_102083/575191149.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X_train[['sent_vec_'+str(i) for i in range(100)]] = pd.DataFrame(glove_vecs_train.tolist())
/tmp/ipykernel_102083/575191149.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of callin

## DeepLearning

# Modeling

In [36]:
from sklearn.metrics import log_loss, accuracy_score
from xgboost import XGBClassifier

## XGBoost

- 숫자형 feature만 선택
- 사용 O: punc, stop_word, num_words, NB 확률 feature, GloVe sent_vec
- 사용 X: text, words, id

In [37]:
X_train.select_dtypes(include='object').columns

Index(['text', 'words'], dtype='object')

In [38]:
drop_cols = ['text', 'words', 'author']

X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
X_xgb = X_train_xgb.drop(columns=drop_cols, errors='ignore')
X_test_xgb = X_test_xgb.drop(columns=drop_cols, errors='ignore')
y_xgb = Y_train.copy()

In [39]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_xgb,
    y_xgb,
    test_size=0.2,
    random_state=42,
    stratify=y_xgb
)

xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=5,
    n_estimators=500,
    learning_rate=0.03,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42
)

xgb.fit(X_tr, y_tr)

val_pred = xgb.predict_proba(X_val)

print("log_loss:", log_loss(y_val, val_pred))
print("accuracy:", accuracy_score(y_val, val_pred.argmax(axis=1)))

log_loss: 0.45341634572670353
accuracy: 0.8340925655976676


## LGBM

In [40]:
from lightgbm import LGBMClassifier

In [42]:
drop_cols = ['text', 'words', 'author']

X_train_lgbm = X_train.copy()
X_test_lgbm = X_test.copy()
X_lgbm = X_train_lgbm.drop(columns=drop_cols, errors='ignore')
X_test_lgbm = X_test_lgbm.drop(columns=drop_cols, errors='ignore')
y_lgbm = Y_train.copy()

In [43]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_lgbm,
    y_lgbm,
    test_size=0.2,
    random_state=42,
    stratify=y_lgbm
)

lgbm = LGBMClassifier(
    objective='multiclass',
    num_class=5,
    n_estimators=500,
    learning_rate=0.03,
    random_state=42
)

lgbm.fit(X_tr, y_tr)

val_pred = lgbm.predict_proba(X_val)

print("log_loss:", log_loss(y_val, val_pred))
print("accuracy:", accuracy_score(y_val, val_pred.argmax(axis=1)))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.077698 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 33778
[LightGBM] [Info] Number of data points in the train set: 43903, number of used features: 136
[LightGBM] [Info] Start training from score -1.422261
[LightGBM] [Info] Start training from score -2.027925
[LightGBM] [Info] Start training from score -1.558116
[LightGBM] [Info] Start training from score -1.292918
[LightGBM] [Info] Start training from score -1.950362
log_loss: 0.45224744028148517
accuracy: 0.8355502915451894


In [ ]:
final_lgbm = LGBMClassifier(
    objective='multiclass',
    num_class=5,
    n_estimators=500,
    learning_rate=0.03,
    random_state=42
)

final_lgbm.fit(X_lgbm, y_lgbm)

lgbm_pred = final_lgbm.predict_proba(X_test_lgbm)